In [2]:
import pickle
import matplotlib.pyplot as plt
import torch
import numpy as np
import seaborn as sns
sns.set()
sns.set_theme(style='whitegrid')

In [4]:
display_name = {'FedAvg': 'LS(FedAvg)', 'Tche': 'TCH', 'AFL': 'OMDgd-TCH(AFL)', 'AFL_new': 'AdaOMDgd-TCH', 
                'EPO': 'EPO', 'FERERO': 'FERERO', 'ExcessMTL': 'ExcessMTL', 'STche': 'STCH'}
seeds = [0, 25, 37, 42, 53, 81, 119, 1010, 1201, 2003]

In [5]:
from itertools import combinations

def calculate_hypervolume_2d(points, ref):
    """
    Calculate the 2D hypervolume for a set of three points (minimization)
    relative to a reference point, excluding overlaps.

    Args:
        points (list of tuple): List of three (x, y) points.
        ref (tuple): Reference point (ref_x, ref_y), must dominate all points.

    Returns:
        float: The hypervolume.
    """
    ref_x, ref_y = ref

    # Helper to compute the dominated rectangle area of given points subset
    def rect_area(subset):
        # For minimization: the *worst* (max) coordinate in the subset determines the overlap origin
        max_x = max(p[0] for p in subset)
        max_y = max(p[1] for p in subset)
        dx = max(0, ref_x - max_x)
        dy = max(0, ref_y - max_y)
        return dx * dy

    # Sum areas of individual rectangles
    hv = sum((ref_x - x) * (ref_y - y) for x, y in points)
    # print(hv)

    # Subtract pairwise intersections
    for pair in combinations(points, 2):
        # print(rect_area(pair))
        hv -= rect_area(pair)

    # Add back the triple intersection
    hv += rect_area(points)

    return hv

In [6]:
import pandas as pd
def get_table(ds_name, allocation_schemes, seeds, methods):
    tb = dict()
    for method in methods:
        data = []
        for seed in seeds:
            points = []
            for allocation_scheme in allocation_schemes:
                path = f'../output/10seeds/{ds_name}_{allocation_scheme}/seed{seed}/{method}/log.pickle'
                with open(path, 'rb') as file:
                    res = pickle.load(file)
                points.append([res[-1]['test_infer_stats']['losses'][0], res[-1]['test_infer_stats']['losses'][1]])
            data.append(calculate_hypervolume_2d(points, [4,4]))
        tb[method] = [np.mean(data), np.std(data)]
        print(f"{method}: {tb[method][0]}")
    

    avg, std = [], []
    for method in methods:
        avg.append(tb[method][0])
        std.append(tb[method][1])
    
    df_avg = pd.DataFrame([avg], index=['Loss'], columns=methods)

    separator = '\t\t'
    # Print the header (column names) with the custom separator
    header = separator.join(df_avg.columns)
    print(f'Index\t{header}')

    formatted_row = separator.join(f'{avg_val:.3f}±{std_val:.3f}' for avg_val, std_val in zip(avg, std))
    print(f'Loss\t{formatted_row}')

In [8]:
get_table('CIFAR10', ['rotation_m1', 'rotation_m2', 'rotation_m3'], seeds, 
              ['FedAvg', 'Tche', 'STche_gamma0.01', 'ExcessMTL_llr1.0', 'EPO', 'FERERO',
               'AFL_llr1.0', 'AFL_new_llr0.03'])

FedAvg: 6.911203644810805
Tche: 6.692914382724979
STche_gamma0.01: 7.582613681248983
ExcessMTL_llr1.0: 8.796801077878676
EPO: 7.456795267522665
FERERO: 6.040880668605191
AFL_llr1.0: 9.166227741113245
AFL_new_llr0.03: 7.00513643697213
Index	FedAvg		Tche		STche_gamma0.01		ExcessMTL_llr1.0		EPO		FERERO		AFL_llr1.0		AFL_new_llr0.03
Loss	6.911±0.092		6.693±0.863		7.583±0.074		8.797±0.052		7.457±0.208		6.041±0.224		9.166±0.065		7.005±0.106


In [11]:
def get_table2(ds_name, allocation_schemes, seeds, methods):
    tb = dict()
    for method in methods:
        data = []
        for seed in seeds:
            points = []
            for allocation_scheme in allocation_schemes:
                path = f'../output/10seeds/{ds_name}_{allocation_scheme}/seed{seed}/{method}/log.pickle'
                with open(path, 'rb') as file:
                    res = pickle.load(file)
                points.append([-res[-1]['test_infer_stats']['accuracys'][0], -res[-1]['test_infer_stats']['accuracys'][1]])
                # print(points[-1])
            data.append(calculate_hypervolume_2d(points, [-0.3,-0.3]))
        tb[method] = [np.mean(data), np.std(data)]
        print(f"{method}: {tb[method][0]}")
    
    avg, std = [], []
    for method in methods:
        avg.append(tb[method][0])
        std.append(tb[method][1])
    
    df_avg = pd.DataFrame([avg], index=['Accuracy'], columns=methods)

    separator = '\t\t'
    # Print the header (column names) with the custom separator
    header = separator.join(df_avg.columns)
    print(f'Index\t{header}')

    formatted_row = separator.join(f'{avg_val:.3f}±{std_val:.3f}' for avg_val, std_val in zip(avg, std))
    print(f'Accuracy\t{formatted_row}')

In [14]:
get_table2('CIFAR10', ['rotation_m1', 'rotation_m2', 'rotation_m3'], seeds, 
              ['FedAvg', 'Tche', 'STche_gamma0.01', 'ExcessMTL_llr1.0', 'EPO', 'FERERO',
               'AFL_llr1.0', 'AFL_new_llr0.03'])

FedAvg: 0.15637833333333337
Tche: 0.12545058666666667
STche_gamma0.01: 0.1465057066666667
ExcessMTL_llr1.0: 0.15329816000000004
EPO: 0.14144177333333335
FERERO: 0.13550232000000004
AFL_llr1.0: 0.16813895999999998
AFL_new_llr0.03: 0.15855592000000004
Index	FedAvg		Tche		STche_gamma0.01		ExcessMTL_llr1.0		EPO		FERERO		AFL_llr1.0		AFL_new_llr0.03
Accuracy	0.156±0.001		0.125±0.019		0.147±0.002		0.153±0.001		0.141±0.004		0.136±0.004		0.168±0.001		0.159±0.002
